# Exercise: Feature Engineering on a Real Dataset

## Predicting Diamond Prices

The snowman data had 14 rows. In this exercise you work with a real dataset of almost 54,000 diamonds. Your goal is to predict the **price** of a diamond from its weight, its dimensions and three quality grades.

You follow the [machine learning workflow](machine_learning_workflow.md) from start to end, in the same steps as the diabetes project: split the data into training, validation and test sets, compare several versions of a model on the validation set, and score only the chosen version on the test set. Each version adds one feature engineering idea, so the validation set tells you which ideas help.

<img src="assets/ml_workflow.png" width="600">

## Learning Objectives

At the end of this exercise, you should be able to:

- Split a dataset into training, validation and test sets, and explore only the training set.
- Encode ordered quality grades with `OrdinalEncoder` and with `OneHotEncoder`, and compare the two on a validation set.
- Log-transform skewed features and a skewed target, and convert the predictions back to the original units.
- Retrain the chosen version on the training and validation sets and score it once on the test set.

## Setup

Documentation: [`set_config`](https://scikit-learn.org/stable/modules/generated/sklearn.set_config.html).

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import RFE
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    PolynomialFeatures,
    StandardScaler,
)

sns.set_style("whitegrid")

# Transformers return pandas DataFrames instead of NumPy arrays
set_config(transform_output="pandas")

## Step 1: Define the Goal and the Metric

**Goal:** predict the price of a diamond in US dollars.

**Metric:** the root mean squared error ([`root_mean_squared_error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html)), in dollars. It reads as "on average, the prediction is about this many dollars off". We also report $R^2$. A model is only useful if it clearly beats a **baseline** that predicts the average price of the training diamonds for every diamond.

## Step 2: Get the Data

The data is the `diamonds` dataset that ships with the R package [ggplot2](https://ggplot2.tidyverse.org/reference/diamonds.html).

| Column | Description |
|---|---|
| `price` | price in US dollars (the target) |
| `carat` | weight of the diamond (1 carat = 0.2 g) |
| `cut` | quality of the cut, from worst to best: `Fair`, `Good`, `Very Good`, `Premium`, `Ideal` |
| `color` | colour grade, from worst to best: `J`, `I`, `H`, `G`, `F`, `E`, `D` |
| `clarity` | clarity grade, from worst to best: `I1`, `SI2`, `SI1`, `VS2`, `VS1`, `VVS2`, `VVS1`, `IF` |
| `depth` | total depth as a percentage of the average width |
| `table` | width of the flat top as a percentage of the widest point |
| `x`, `y`, `z` | length, width and depth in mm |

`cut`, `color` and `clarity` are **ordinal**: their categories have a natural order.

In [ ]:
df = pd.read_csv("data/diamonds.csv")
df.head()

In [ ]:
df.info()

## Step 3: Split into Training, Validation and Test Sets

Split **before** you look closely at the data, exactly as in the diabetes project: first set aside 33% of the diamonds as the test set, then 25% of the rest as the validation set. Use `random_state=42` so your numbers match the solution. There is no `stratify` here, because the target is a number, not a class.

Documentation: [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).

In [ ]:
# TODO: first split: set the test set aside (test_size=0.33)
df_train_val, df_test = train_test_split(...)

# TODO: second split: separate the validation set (test_size=0.25)
df_train, df_val = ...

print(f"Training set:   {df_train.shape[0]} rows")
print(f"Validation set: {df_val.shape[0]} rows")
print(f"Test set:       {df_test.shape[0]} rows")

## Step 4: Explore the Training Set

Explore only `df_train`. What you find here guides your feature engineering.

`df.info()` showed no missing values. Look at the summary statistics of the numeric columns: does every value make sense for a real diamond?

In [ ]:
# TODO: summary statistics of the numeric columns of df_train
...

Count the training diamonds with a zero in `x`, `y` or `z`.

In [ ]:
# TODO: count the rows where any of x, y, z equals 0
#       hint: (df_train[["x", "y", "z"]] == 0).any(axis=1)
...

Plot the distribution of `price`. Is it symmetric?

In [ ]:
# TODO: a histogram of the price
sns.histplot(...);

Plot `price` against `carat`. The training set has about 27,000 points, so make them small and transparent (`s=5`, `alpha=0.1`). Is the relationship a straight line?

In [ ]:
# TODO: a scatter plot of carat (x) against price (y)
sns.scatterplot(data=df_train, x=..., y=..., s=5, alpha=0.1);

Calculate the median price for each `cut`. Which cut is the cheapest, and why could that be? Hint: compare the median `carat` of each cut as well.

In [ ]:
# TODO: the median price and the median carat for each cut
df_train.groupby("cut")[[..., ...]].median()

## Step 5: Mark the Impossible Values as Missing

A zero length, width or depth is a recording error, not a real measurement. Replace the zeros in `x`, `y` and `z` with missing values (`np.nan`) in **all three** sets, so that an imputer can fill them later.

This replacement does not leak information: it applies a fixed rule to each row and learns nothing from the data. The imputer **learns** a median, so it belongs in the pipeline and is fitted on the training set only.

In [ ]:
def zeros_to_missing(data):
    """Return a copy of data with the zeros in x, y and z replaced by np.nan."""
    data = data.copy()
    # TODO: replace 0 with np.nan in the three dimension columns
    data[["x", "y", "z"]] = ...
    return data


df_train = zeros_to_missing(df_train)
df_val = zeros_to_missing(df_val)
df_test = zeros_to_missing(df_test)
df_train[["x", "y", "z"]].isna().sum()

Separate the features from the target in each set.

In [ ]:
target = "price"

X_train, y_train = df_train.drop(columns=target), df_train[target]
X_val, y_val = df_val.drop(columns=target), df_val[target]
X_test, y_test = df_test.drop(columns=target), df_test[target]

## Steps 6 to 8: Train and Compare Model Versions

Every version is a [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) with a [`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) followed by a [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html). Each version changes one feature engineering idea. Before you build the versions, complete the helper `evaluate`, which you will call once for every version. The `else` branch, for a price in dollars, is already written: use it as the model for the `log_target` branch. The helper should:

1. Fit the model on the training set.
2. Predict the training **and** the validation set, so you can compare the two scores and spot overfitting.
3. Calculate the RMSE on both sets and $R^2$ on the validation set.
4. Report the **lowest** predicted price: a price below zero is a sign that the model does not fit the data well.

When `log_target=True`, the model must learn the **logarithm** of the price: fit it on `np.log(y_train)`, then convert its predictions back to dollars with `np.exp`, so that every score is in dollars. You will need this in version 4.

To check your helper: version 1 should reach a validation RMSE of about 1,487 dollars.

Documentation: [`r2_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html).

In [ ]:
scores = []


def evaluate(name, model, log_target=False):
    """Fit model on the training set, score it in dollars on the training and validation sets."""
    if log_target:
        # TODO: as in the else branch below, but fit on np.log(y_train)
        model.fit(X_train, ...)
        # TODO: convert the predictions back to dollars with np.exp
        train_pred = np.exp(...)
        val_pred = ...
    else:
        model.fit(X_train, y_train)
        train_pred = model.predict(X_train)
        val_pred = model.predict(X_val)
    scores.append(
        {
            "version": name,
            "train RMSE": root_mean_squared_error(y_train, train_pred),
            # TODO: the RMSE and R2 on the validation set
            "validation RMSE": root_mean_squared_error(..., ...),
            "validation R2": r2_score(..., ...),
            "lowest prediction": val_pred.min(),
        }
    )
    return pd.DataFrame(scores).round(3)

### Baseline

The baseline predicts the training mean for every diamond. Its RMSE is the number every model must beat.

In [ ]:
baseline_pred = pd.Series(y_train.mean(), index=y_val.index)
root_mean_squared_error(y_val, baseline_pred)

### Version 1: Numeric Features

Use the six numeric columns. `x`, `y` and `z` now contain missing values, so every numeric column goes through a pipeline that imputes the median and then standardises.

Documentation: [`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html), [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html).

In [ ]:
numeric = ["carat", "depth", "table", "x", "y", "z"]

# TODO: impute with the median, then StandardScaler
numeric_pipeline = Pipeline(steps=[("imputer", ...), ("scaler", ...)])

v1 = Pipeline(
    steps=[
        ("features", ColumnTransformer(transformers=[("numeric", numeric_pipeline, numeric)])),
        ("model", LinearRegression()),
    ]
)
evaluate("1 numeric", v1)

### Version 2: Add the Grades with `OrdinalEncoder`

The three grades are ordinal, so encode them with [`OrdinalEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html). Pass the order of each grade, from worst to best, with the `categories` argument: otherwise the encoder sorts the categories alphabetically, and `Fair < Good < Ideal < Premium < Very Good` is not the order of the cut.

In [ ]:
grades = ["cut", "color", "clarity"]
grade_order = [
    ["Fair", "Good", "Very Good", "Premium", "Ideal"],
    ["J", "I", "H", "G", "F", "E", "D"],
    ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"],
]

# TODO: an OrdinalEncoder that uses grade_order
ordinal_encoder = ...

# TODO: add a second transformer that applies ordinal_encoder to the grades
v2 = Pipeline(
    steps=[
        (
            "features",
            ColumnTransformer(
                transformers=[
                    ("numeric", numeric_pipeline, numeric),
                    ...,
                ]
            ),
        ),
        ("model", LinearRegression()),
    ]
)
evaluate("2 + ordinal grades", v2)

### Version 3: One-Hot Encode the Grades Instead

`OrdinalEncoder` turns each grade into 0, 1, 2, ..., so the linear model gives every step up the scale the **same** price change. One-hot encoding lets each grade have its own price change instead. Replace the ordinal encoder with a [`OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

In [ ]:
# TODO: a OneHotEncoder with drop="first", handle_unknown="ignore", sparse_output=False
onehot_encoder = ...

# TODO: version 2 with onehot_encoder in place of ordinal_encoder
v3 = ...
evaluate("3 one-hot grades", v3)

### Version 4: Log-Transform the Skewed Columns

Your plots showed a skewed price and a curved relationship with `carat`. Taking the logarithm of both straightens the curve and stops the model from predicting negative prices, because the exponential of any number is positive.

- Give `carat`, `x`, `y` and `z` a pipeline that imputes, applies `np.log` with a [`FunctionTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.FunctionTransformer.html), then scales.
- Keep `depth` and `table` in the numeric pipeline, and keep the one-hot encoded grades.
- Call `evaluate` with `log_target=True`, so the model learns the logarithm of the price.

In [ ]:
skewed = ["carat", "x", "y", "z"]
other = ["depth", "table"]

# TODO: impute (median), FunctionTransformer(func=np.log, feature_names_out="one-to-one"), then StandardScaler
#       (feature_names_out keeps the column names, which you need in the bonus)
log_pipeline = ...

# TODO: three transformers: log_pipeline on skewed, numeric_pipeline on other, onehot_encoder on grades
v4 = ...
evaluate("4 + log transform", v4, log_target=True)

### Choose a Version

Which version has the lowest validation RMSE? Is its training RMSE close to its validation RMSE, or does it overfit? Does it ever predict a negative price? Write down your choice before you continue.

## Step 9: Calculate the Test Score

As in the diabetes project, retrain the chosen version on the training and validation sets **together**, with the same settings, then score it **once** on the test set. Remember to train on the logarithm of the price and to convert the predictions back with `np.exp`. Score the baseline on the test set too, for comparison.

In [ ]:
X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])

# TODO: fit your chosen version on X_train_val and np.log(y_train_val)
...

# TODO: predict the test set, convert back to dollars, and calculate the RMSE and R2
test_pred = ...
print(f"Test RMSE: {...:.0f} dollars  Test R2: {...:.3f}")

baseline_test = pd.Series(y_train_val.mean(), index=y_test.index)
print(f"Baseline test RMSE: {root_mean_squared_error(y_test, baseline_test):.0f} dollars")

## Bonus: Polynomial Terms Instead of the Logarithm

[`PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) is another way to follow a curve. Build a version 5 from version 3 (no log transform, the price in dollars), but give `carat` its own pipeline that imputes, scales and adds degree-3 polynomial terms. Compare it with version 4 on the validation set, including the lowest prediction.

Use the training and validation sets only: the test set has already been used for the final score.

In [ ]:
# TODO: impute (median), StandardScaler, then PolynomialFeatures(degree=3, include_bias=False)
polynomial_pipeline = ...

# TODO: polynomial_pipeline on carat, numeric_pipeline on the other numeric columns, onehot_encoder on grades
v5 = ...
evaluate("5 polynomial carat", v5)

## Bonus 2: Feature Selection

Version 4 uses 23 columns. In W3D2 you learned to ask whether all of them are needed. The question matters here because `carat`, `x`, `y` and `z` all measure the **size** of a diamond.

Start with a filter method: fit the `ColumnTransformer` of version 4 on the training set and look at the correlations between its six numeric output columns.

In [ ]:
# TODO: fit_transform X_train with the ColumnTransformer of version 4 (v4.named_steps["features"])
features_train = ...

# TODO: the correlation matrix of the first six columns (the numeric features), as a heatmap
sns.heatmap(features_train.iloc[:, :6].corr(), annot=True, fmt=".3f", cmap="coolwarm", center=0);

Compare two smaller versions with version 4 on the validation set. Both keep the log-transformed `carat`, the one-hot encoded grades and the log target:

- **Version 6** drops `x`, `y` and `z`.
- **Version 7** also drops `depth` and `table`, so it keeps only `carat` and the grades.

Then test whether the grades are needed at all with **version 8**: version 4 without the grades.

In [ ]:
# TODO: version 6: log_pipeline on ["carat"], numeric_pipeline on other, onehot_encoder on grades
v6 = ...
evaluate("6 without x, y, z", v6, log_target=True)

# TODO: version 7: log_pipeline on ["carat"], onehot_encoder on grades
v7 = ...
evaluate("7 carat and grades only", v7, log_target=True)

# TODO: version 8: version 4 without the grades transformer
v8 = ...
evaluate("8 without grades", v8, log_target=True)

Finally, try a wrapper method. [`RFE`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html) can sit inside the pipeline, between the `ColumnTransformer` and the model, and keep the 10 output columns it ranks highest. Compare its validation RMSE with version 7, which uses 18 columns (`carat` and the 17 one-hot grade columns), and look at which columns RFE kept.

In [ ]:
# TODO: a pipeline: the ColumnTransformer of version 4, then
#       ("selector", RFE(LinearRegression(), n_features_to_select=10)), then the model
v9 = ...
evaluate("9 RFE, 10 columns", v9, log_target=True)

In [ ]:
# The columns RFE kept
v9.named_steps["selector"].get_feature_names_out()

### Create a Feature by Hand: Volume

Instead of dropping `x`, `y` and `z`, you can **combine** them into one feature that has a physical meaning: the volume of the diamond in cubic millimetres, `x * y * z`. Add it to the training and the validation set (the test set has already been used). As in Step 5, this is a fixed rule for each row, so it learns nothing and leaks nothing.

How strongly is the logarithm of the volume correlated with the logarithm of `carat`?

In [ ]:
for X in (X_train, X_val):
    # TODO: the volume in cubic millimetres: x times y times z
    X["volume"] = ...

# TODO: the correlation between the log of volume and the log of carat in the training set
np.log(X_train["volume"]).corr(...)

Weight and volume measure the same thing, so the volume could replace `carat`. Build **version 10**: version 7 with `volume` in place of `carat`, and compare. Look at the training RMSE as well as the validation RMSE.

In [ ]:
# TODO: log_pipeline on ["volume"], onehot_encoder on grades
v10 = ...
evaluate("10 volume instead of carat", v10, log_target=True)

A training RMSE far **above** the validation RMSE is unusual. Find the training diamond with the largest error and look at its dimensions.

In [ ]:
train_pred = pd.Series(np.exp(v10.predict(X_train)), index=X_train.index)

# TODO: the absolute error of each training diamond
errors = ...

# TODO: show carat, x, y, z, volume and the price of the diamond with the largest error
#       hint: errors.idxmax() gives its index
X_train.loc[[...], ["carat", "x", "y", "z", "volume"]].assign(price=y_train, predicted=train_pred)

## Summary

In this exercise you:

- Split a real dataset into training, validation and test sets, and explored only the training set.
- Marked impossible values as missing and imputed them inside a pipeline fitted on the training set.
- Compared ordinal and one-hot encoding of quality grades on the validation set.
- Log-transformed skewed features and the target, which improved the model and removed negative predictions.
- Retrained the chosen version on the training and validation sets and scored it once on the test set.
- Selected features as whole groups, created a volume feature by hand, and compared these versions on the validation set (bonus).

With about 9,000 validation diamonds, the comparison of versions is already stable. On smaller datasets it is not, which is why cross-validation in W3D4 replaces the single validation set.